# Friend vs. Enemy detection (POC)



For each of the **top-200 characters** (by degree in the affiliation LCC), we send their bio to **Claude Haiku** along with the list of their `affiliated` IDs (filtered to the same top-200), and ask the model to score every relationship from that character's perspective on a 0–100 scale:



| Score | Meaning |

|---|---|

| 0 | Sworn enemy |

| 25 | Rival / adversary |

| 50 | Neutral / acquainted |

| 75 | Ally / friend |

| 100 | Closest bond / love / unwavering loyalty |



The result is a **directed** graph: A→B is A's view of B. Most edges point the same way as B→A in practice, so the directionality is rarely visible — but it lets us catch one-sided relationships (e.g. Tyrion → Tywin = enemy, but Tywin → Tyrion may be 'merely useful').



**Cost**: ~200 Haiku calls, each with one bio + a short list. Estimated **<$2** total.

## 1. Setup

In [ ]:
import os, json, time, hashlib
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import Counter, defaultdict

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
from tqdm import tqdm

from dotenv import load_dotenv
import anthropic

load_dotenv()
if not os.environ.get('ANTHROPIC_API_KEY'):
    raise RuntimeError('ANTHROPIC_API_KEY not set. Drop it into the .env file next to this notebook.')

client = anthropic.Anthropic()
MODEL = 'claude-haiku-4-5-20251001'

CACHE_DIR = Path('relationships_cache')
CACHE_DIR.mkdir(exist_ok=True)
print(f'Cache dir: {CACHE_DIR.resolve()}')

## 2. Load data and pick top-200 characters



Top-200 by degree in the LCC of the affiliation graph (the same graph as the other notebooks).

In [ ]:
TOP_N = 200

df = pd.read_csv('../csvs/characters_enriched.csv').fillna('')
bio_df = pd.read_csv('../csvs/characters_bio.csv').fillna('')

valid_ids = set(df['ID'])
edges = set()
for _, row in df.iterrows():
    src = row['ID']
    if not row['affiliated']:
        continue
    for tgt in row['affiliated'].split(';'):
        tgt = tgt.strip()
        if tgt and tgt != src and tgt in valid_ids:
            edges.add(frozenset({src, tgt}))

G = nx.Graph()
G.add_nodes_from(df['ID'])
G.add_edges_from(tuple(e) for e in edges)
LCC = G.subgraph(max(nx.connected_components(G), key=len)).copy()

degrees = dict(LCC.degree())
top_ids = [n for n, _ in sorted(degrees.items(), key=lambda x: -x[1])[:TOP_N]]
top_set = set(top_ids)

name_by_id = dict(zip(df['ID'], df['name']))
primary_house = {
    row['ID']: (row['allegiance'].split(';')[0].strip() if row['allegiance'] else '')
    for _, row in df.iterrows()
}
bio_by_id = dict(zip(bio_df['ID'], bio_df['bio']))

print(f'Top {TOP_N} characters by degree')
print(f'With bio:    {sum(1 for i in top_ids if bio_by_id.get(i, "").strip())} / {TOP_N}')
print(f'\nTop 10 by degree:')
for i in top_ids[:10]:
    print(f'  {degrees[i]:>4}  {name_by_id[i]}')

### Build candidate edge list



For each top-200 character, list of their `affiliated` IDs that are also in top-200. This is what we'll send to Haiku.

In [ ]:
row_by_id = {row['ID']: row for _, row in df.iterrows()}

candidate_targets = {}
for src in top_ids:
    aff = row_by_id[src]['affiliated']
    if not aff:
        candidate_targets[src] = []
        continue
    targets = []
    seen = set()
    for tgt in aff.split(';'):
        tgt = tgt.strip()
        if tgt and tgt != src and tgt in top_set and tgt not in seen:
            targets.append(tgt)
            seen.add(tgt)
    candidate_targets[src] = targets

edge_counts = [len(v) for v in candidate_targets.values()]
print(f'Total directed candidate edges: {sum(edge_counts)}')
print(f'Avg per character: {np.mean(edge_counts):.1f}')
print(f'Max per character: {max(edge_counts)}')
print(f'Characters with 0 candidates (will be skipped): {sum(1 for c in edge_counts if c == 0)}')

## 3. The LLM call — structured tool output



We use Anthropic's tools API with a forced `submit_relationships` schema, which guarantees a list of `{target_id, score, confidence, evidence}` objects without needing to parse free text.

In [ ]:
MAX_BIO_CHARS = 25_000  # ≈ 6k tokens; cap so single huge bios don't blow up cost

SYSTEM_PROMPT = (
    'You are an expert reader of George R. R. Martin\u2019s A Song of Ice and Fire / Game of Thrones. '
    'You read character biographies and judge relationships based on the narrative.\n\n'
    'Scoring scale (0\u2013100, the SUBJECT character\u2019s view of the OTHER):\n'
    '  0   = sworn enemy / mortal hatred / would kill on sight\n'
    '  25  = rival, adversary, or open dislike\n'
    '  50  = neutral, indifferent, brief acquaintance, or no real interaction\n'
    '  75  = ally, friend, or trusted companion\n'
    '  100 = closest bond, love, sworn loyalty, or family devotion\n\n'
    'Important rules:\n'
    '\u2022 Family ties do not imply friendship. Tyrion despises Tywin even though they\u2019re father and son. Score the actual emotional/political relationship, not blood.\n'
    '\u2022 Bios may use first names, titles, or nicknames. Match by context.\n'
    '\u2022 If the bio gives no real information about a relationship, score 50 and set confidence to 0.0.\n'
    '\u2022 Confidence reflects how clearly the bio establishes the relationship (1.0 = explicit and consistent, 0.0 = no evidence in the bio).\n'
    '\u2022 evidence should be a short paraphrase or quote (\u226430 words) supporting the score.'
)

TOOL = {
    'name': 'submit_relationships',
    'description': 'Submit a relationship score for every requested target.',
    'input_schema': {
        'type': 'object',
        'properties': {
            'relationships': {
                'type': 'array',
                'items': {
                    'type': 'object',
                    'properties': {
                        'target_id': {'type': 'string', 'description': 'The id from the request list.'},
                        'score':      {'type': 'integer', 'minimum': 0, 'maximum': 100},
                        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
                        'evidence':   {'type': 'string'},
                    },
                    'required': ['target_id', 'score', 'confidence', 'evidence'],
                },
            },
        },
        'required': ['relationships'],
    },
}

In [ ]:
def build_user_message(subject_id: str, subject_name: str, bio: str, targets: list[str]) -> str:
    bio_clipped = bio[:MAX_BIO_CHARS]
    truncated_note = '' if len(bio) <= MAX_BIO_CHARS else f'\n\n[bio truncated at {MAX_BIO_CHARS} chars]'
    target_lines = '\n'.join(
        f'{i+1}. {name_by_id.get(t, t)} (id: {t})' for i, t in enumerate(targets)
    )
    return (
        f'SUBJECT: {subject_name} (id: {subject_id})\n\n'
        f'BIOGRAPHY OF {subject_name}:\n{bio_clipped}{truncated_note}\n\n'
        f'For each of the following characters, score the relationship from {subject_name}\u2019s perspective.\n'
        f'Return one entry per id, using the exact id strings shown:\n\n'
        f'{target_lines}\n\n'
        f'Call the submit_relationships tool with one object per target.'
    )

In [ ]:
def cache_path(subject_id: str) -> Path:
    return CACHE_DIR / f'{subject_id}.json'

def score_character(subject_id: str, max_retries: int = 3) -> dict | None:
    """Score every candidate target for one subject character.

    Returns a dict {'subject': id, 'relationships': [...]} or None if skipped/failed.
    Cached on disk so reruns are free.
    """
    cp = cache_path(subject_id)
    if cp.exists():
        return json.loads(cp.read_text())

    targets = candidate_targets[subject_id]
    if not targets:
        result = {'subject': subject_id, 'relationships': [], 'skipped': 'no_targets'}
        cp.write_text(json.dumps(result))
        return result

    bio = bio_by_id.get(subject_id, '').strip()
    if not bio:
        result = {'subject': subject_id, 'relationships': [], 'skipped': 'no_bio'}
        cp.write_text(json.dumps(result))
        return result

    user_msg = build_user_message(subject_id, name_by_id[subject_id], bio, targets)

    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.messages.create(
                model=MODEL,
                max_tokens=4096,
                system=[{
                    'type': 'text',
                    'text': SYSTEM_PROMPT,
                    'cache_control': {'type': 'ephemeral'},
                }],
                tools=[TOOL],
                tool_choice={'type': 'tool', 'name': 'submit_relationships'},
                messages=[{'role': 'user', 'content': user_msg}],
            )
            tool_use = next((b for b in resp.content if b.type == 'tool_use'), None)
            if tool_use is None:
                raise RuntimeError('no tool_use block in response')
            rels = tool_use.input['relationships']
            # Validate target ids match what we asked for
            requested = set(targets)
            returned = {r['target_id'] for r in rels}
            missing = requested - returned
            for m in missing:  # fill gaps with neutral / no evidence
                rels.append({'target_id': m, 'score': 50, 'confidence': 0.0,
                             'evidence': '[not returned by model]'})
            result = {
                'subject': subject_id,
                'relationships': rels,
                'usage': {'input_tokens': resp.usage.input_tokens,
                          'output_tokens': resp.usage.output_tokens},
            }
            cp.write_text(json.dumps(result))
            return result
        except Exception as e:
            last_err = e
            time.sleep(2 ** attempt)
    print(f'FAILED {subject_id}: {last_err}')
    return None

### Quick smoke test on a single character



Run this once first to confirm the API key, the prompt, and the tool schema all work end-to-end before you fire off all 200 calls.

In [ ]:
smoke_id = top_ids[0]  # highest-degree character
print(f'Smoke testing on: {name_by_id[smoke_id]}  ({len(candidate_targets[smoke_id])} candidates)')
result = score_character(smoke_id)
if result:
    print(f'\nFirst 5 relationships for {name_by_id[smoke_id]}:')
    for r in result['relationships'][:5]:
        tn = name_by_id.get(r["target_id"], r["target_id"])
        print(f'  -> {tn:<25}  score={r["score"]:>3}  conf={r["confidence"]:.2f}  | {r["evidence"][:80]}')

## 4. Run on all top-200 (parallelised)



Cached responses are skipped instantly. Press the cell again any time to resume after a failure.

In [ ]:
MAX_WORKERS = 8  # gentle on rate limits; bump up if you have higher tier

to_run = [i for i in top_ids if not cache_path(i).exists()]
print(f'{len(to_run)} characters left to score (out of {TOP_N})')

if to_run:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(score_character, i): i for i in to_run}
        for fut in tqdm(as_completed(futures), total=len(futures), desc='scoring'):
            _ = fut.result()

print('Done. All cached.')

In [ ]:
# Aggregate token usage / cost estimate
all_results = []
for i in top_ids:
    cp = cache_path(i)
    if cp.exists():
        all_results.append(json.loads(cp.read_text()))

in_tok = sum(r.get('usage', {}).get('input_tokens', 0) for r in all_results)
out_tok = sum(r.get('usage', {}).get('output_tokens', 0) for r in all_results)
# Haiku 4.5 list price: ~$1/M input, ~$5/M output (verify on console)
cost = in_tok / 1_000_000 * 1.0 + out_tok / 1_000_000 * 5.0
print(f'Total input tokens : {in_tok:>10,}')
print(f'Total output tokens: {out_tok:>10,}')
print(f'Approx. cost      : ${cost:.2f}  (verify against your dashboard)')

## 5. Build the directed scored graph

In [ ]:
DG = nx.DiGraph()
for i in top_ids:
    DG.add_node(i, name=name_by_id[i], house=primary_house.get(i, ''))

for r in all_results:
    src = r['subject']
    for rel in r.get('relationships', []):
        tgt = rel['target_id']
        if tgt not in DG:  # ignore stray ids the model invented
            continue
        DG.add_edge(src, tgt,
                    score=rel['score'],
                    confidence=rel['confidence'],
                    evidence=rel['evidence'])

print(f'Directed graph: {DG.number_of_nodes()} nodes, {DG.number_of_edges()} edges')

scores = [d['score'] for _, _, d in DG.edges(data=True)]
print(f'\nScore distribution:')
print(f'  enemy   (\u2264 33): {sum(1 for s in scores if s <= 33):>4}  ({sum(1 for s in scores if s <= 33)/len(scores):.1%})')
print(f'  neutral (34\u201366): {sum(1 for s in scores if 34 <= s <= 66):>4}  ({sum(1 for s in scores if 34 <= s <= 66)/len(scores):.1%})')
print(f'  friend  (\u2265 67): {sum(1 for s in scores if s >= 67):>4}  ({sum(1 for s in scores if s >= 67)/len(scores):.1%})')
print(f'\nMean score: {np.mean(scores):.1f}, median: {np.median(scores)}')

In [ ]:
# Histogram of scores
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(scores, bins=20, color='#444', edgecolor='white')
ax.axvspan(0, 33, alpha=0.15, color='#d62728', label='enemy')
ax.axvspan(34, 66, alpha=0.10, color='gray', label='neutral')
ax.axvspan(67, 100, alpha=0.15, color='#2ca02c', label='friend')
ax.set_xlabel('relationship score (0=enemy, 100=closest bond)')
ax.set_ylabel('# directed edges')
ax.set_title(f'Distribution of LLM-scored relationships ({DG.number_of_edges()} edges)')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Visualisation



* **Node colour**: primary house (top 15 distinct, rest grey).

* **Edge colour**: continuous gradient red (0) \u2192 grey (50) \u2192 green (100). Confidence drives alpha — uncertain edges are faint.

* **Arrows**: small, since A\u2192B and B\u2192A usually agree.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, Normalize

rel_cmap = LinearSegmentedColormap.from_list(
    'rel', [(0.0, '#d62728'), (0.5, '#bbbbbb'), (1.0, '#2ca02c')]
)
norm = Normalize(vmin=0, vmax=100)

# House colours — top 15 most frequent houses among top-200 nodes
house_counts = Counter(primary_house.get(n, '') for n in DG.nodes() if primary_house.get(n))
top_houses = [h for h, _ in house_counts.most_common(15)]
house_cmap = cm.get_cmap('tab20', len(top_houses))
house_color = {h: house_cmap(i) for i, h in enumerate(top_houses)}

def node_color(n):
    h = primary_house.get(n, '')
    return house_color.get(h, (0.8, 0.8, 0.8, 0.6))

In [ ]:
# Layout once, reuse below
pos = nx.spring_layout(DG.to_undirected(), seed=42, k=0.4, iterations=80)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 14))

# Edges: draw enemies last (so they pop), neutrals first as faint background
edge_data = list(DG.edges(data=True))
edge_data.sort(key=lambda e: abs(e[2]['score'] - 50))  # neutral first, extremes last

for u, v, d in edge_data:
    color = rel_cmap(norm(d['score']))
    alpha = 0.25 + 0.55 * d['confidence']  # 0.25 … 0.80
    nx.draw_networkx_edges(
        DG, pos, edgelist=[(u, v)], ax=ax,
        edge_color=[color], alpha=alpha, width=0.6,
        arrows=True, arrowsize=4, arrowstyle='-|>',
        connectionstyle='arc3,rad=0.05',
    )

# Nodes sized by total degree in DG
deg = dict(DG.degree())
sizes = [40 + 6 * deg[n] for n in DG.nodes()]
colors = [node_color(n) for n in DG.nodes()]
nx.draw_networkx_nodes(DG, pos, node_size=sizes, node_color=colors,
                       linewidths=0.5, edgecolors='white', ax=ax)

# Labels for the 30 highest-degree nodes only
label_ids = sorted(DG.nodes(), key=lambda n: -deg[n])[:30]
labels = {n: name_by_id[n] for n in label_ids}
nx.draw_networkx_labels(DG, pos, labels=labels, font_size=8, font_weight='bold', ax=ax)

# Legends
house_patches = [mpatches.Patch(color=house_color[h], label=h) for h in top_houses]
edge_handles = [
    plt.Line2D([0], [0], color='#d62728', lw=2, label='enemy (\u2264 33)'),
    plt.Line2D([0], [0], color='#bbbbbb', lw=2, label='neutral (34\u201366)'),
    plt.Line2D([0], [0], color='#2ca02c', lw=2, label='friend (\u2265 67)'),
]
leg1 = ax.legend(handles=edge_handles, loc='upper left', title='relationship', fontsize=9)
ax.add_artist(leg1)
ax.legend(handles=house_patches, loc='lower left', title='house', fontsize=7, ncol=2)

ax.set_title(f'Friend / enemy network among the top {TOP_N} characters\n'
             f'edge colour = relationship score (red=enemy, green=friend), node colour = primary house',
             fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.show()

## 7. Per-house and per-character summaries

In [ ]:
# For each character: average score they GIVE (how friendly are they) and RECEIVE (how liked are they)
out_rows = []
for n in DG.nodes():
    out_scores = [d['score'] for _, _, d in DG.out_edges(n, data=True)]
    in_scores  = [d['score'] for _, _, d in DG.in_edges(n,  data=True)]
    out_rows.append({
        'name': name_by_id[n],
        'house': primary_house.get(n, ''),
        'degree': deg[n],
        'avg_outgoing': round(np.mean(out_scores), 1) if out_scores else None,
        'avg_incoming': round(np.mean(in_scores),  1) if in_scores  else None,
    })
char_summary = pd.DataFrame(out_rows).sort_values('degree', ascending=False)
char_summary.head(20)

In [ ]:
# Most disliked / most beloved
ranked = char_summary.dropna(subset=['avg_incoming']).sort_values('avg_incoming')
print('Most disliked (lowest avg incoming score):')
print(ranked[['name', 'house', 'avg_incoming']].head(10).to_string(index=False))
print('\nMost beloved (highest avg incoming score):')
print(ranked[['name', 'house', 'avg_incoming']].tail(10).iloc[::-1].to_string(index=False))

In [ ]:
# Per-house intra vs extra-house friendliness
rows = []
for h in top_houses:
    intra, extra = [], []
    for u, v, d in DG.edges(data=True):
        hu, hv = primary_house.get(u, ''), primary_house.get(v, '')
        if hu == h:
            (intra if hv == h else extra).append(d['score'])
    rows.append({
        'house': h,
        'intra_avg': round(np.mean(intra), 1) if intra else None,
        'extra_avg': round(np.mean(extra), 1) if extra else None,
        'intra_n': len(intra),
        'extra_n': len(extra),
    })
house_summary = pd.DataFrame(rows)
house_summary

## 8. Save outputs

In [ ]:
# Edge table for downstream use / website
edge_rows = []
for u, v, d in DG.edges(data=True):
    edge_rows.append({
        'source':       u,
        'source_name':  name_by_id[u],
        'source_house': primary_house.get(u, ''),
        'target':       v,
        'target_name':  name_by_id[v],
        'target_house': primary_house.get(v, ''),
        'score':        d['score'],
        'confidence':   d['confidence'],
        'evidence':     d['evidence'],
    })
edge_df = pd.DataFrame(edge_rows)
edge_df.to_csv('friend_enemy_edges.csv', index=False)
print(f'Saved {len(edge_df)} directed edges to friend_enemy_edges.csv')

# Node table
node_df = pd.DataFrame([
    {'id': n, 'name': name_by_id[n], 'house': primary_house.get(n, ''), 'degree': deg[n]}
    for n in DG.nodes()
])
node_df.to_csv('friend_enemy_nodes.csv', index=False)
print(f'Saved {len(node_df)} nodes to friend_enemy_nodes.csv')

# JSON for D3 / web visualisations
graph_json = {
    'nodes': node_df.to_dict(orient='records'),
    'links': edge_rows,
}
with open('friend_enemy_graph.json', 'w') as f:
    json.dump(graph_json, f)
print('Saved friend_enemy_graph.json (nodes + links) for the website.')

## 9. Caveats for the report



* **LLM uncertainty.** Every score is Haiku\u2019s best guess from the bio. The `confidence` field captures the model\u2019s self-rating; we render it as edge alpha, so faint edges are the ones to take with a grain of salt. Spot-check the most extreme edges manually before publishing.

* **Disambiguation.** When a bio says \u201cAegon\u201d, Haiku has to pick from several Aegons. We mitigate this by passing only the candidate ids that are *already* in the affiliation list \u2014 so the model isn\u2019t asked which Aegon, only how the relationship reads. Errors will still happen on minor ambiguous mentions.

* **Top-200 only.** A character outside the top 200 cannot appear in this graph even if they have a strong relationship with someone inside it. Bumping the threshold is a one-line change (`TOP_N = 500`) but quadruples the candidate edges.

* **Bios reflect the writer\u2019s framing.** The wiki bios summarise the books, but emphasis varies. A character whose bio dwells on a single feud may look more polarised than one with a balanced summary.

* **Directed but rarely visible.** A\u2192B and B\u2192A usually agree. The directionality matters most for asymmetric cases (Tyrion\u2192Tywin = 0, Tywin\u2192Tyrion may be 30) \u2014 worth scanning `edge_df` for the largest disagreements.
